# Labeling: build your own test set

This notebook builds the **test set** that the cascade is evaluated on later. You draw a box around each object you care about and give it a **label**. The labels are saved to `annotations.json` in the image folder.

**What a label means**
- **label** = the *exact physical item*. Two boxes with the same label say "this is the same object" (the same sofa in two photos).
- **group** (optional) = a coarser identity, such as the *product model*. Two different sofas of the same model get different labels but the same group.

**What ends up in the test set**
- Every box becomes one test item, and metrics are computed per box.
- An image you mark as **reviewed** without drawing boxes becomes a *distractor*: something that should never be returned as a match.
- Images you haven't looked at yet stay out of the test set, so a half-finished labeling session never adds wrong distractors.

Nothing here touches the images themselves; only `annotations.json` (plus a backup, `annotations.json.bak`) is written.

## Setup

This cell switches matplotlib to interactive widgets (needed to draw boxes), imports the labeling app and sets the folder to label.

- **Default folder:** `data/testset/` in the repository (gitignored, so your photos are never committed). Copy your test images there; subfolders are fine.
- **Another folder:** change `TESTSET_DIR`, or set the environment variable `VS_TESTSET` before starting JupyterLab.

In [ ]:
%matplotlib widget
import os
from pathlib import Path

from visual_search.data import split_dataset, validate_dataset
from visual_search.data.schema import parse_dataset
from visual_search.ui import LabelingApp

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
TESTSET_DIR = Path(os.environ.get("VS_TESTSET", ROOT / "data" / "testset"))

if not TESTSET_DIR.is_dir():
    print(f"Create {TESTSET_DIR} and copy your test images into it, then re-run this cell.")
else:
    n = sum(1 for p in TESTSET_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"})
    print(f"Labeling {TESTSET_DIR} ({n} images)")

## Label the images

The app below opens at the first image that hasn't been labelled yet.

**Navigating**
- **Image** dropdown: ✓ marks images that are done, and the number in brackets is the box count.
- **◀ Prev / Next ▶** step through the images; **Next unlabelled** jumps to the next image without labels.

**Adding a box**
1. Drag a rectangle around the object on the image. Use the toolbar's zoom for small objects, then switch zoom off again before drawing.
2. Type a **Label**. The field suggests labels you already used, so the same object keeps exactly the same label across photos.
3. Optionally type a **Group** (e.g. the product model).
4. Press **Add box**. The box is drawn with its label and saved immediately.

**Duplicate check:** if a *new* label looks like an existing one (`Bank_01` vs `bank_01`, `bank_1`), a warning appears and the button changes to **Add anyway**. Either pick the existing label or press it again if it really is a different object.

**Fixing mistakes:** select a box in the **Boxes** list. It's highlighted, and its label fills the fields. Then use **Relabel selected** (applies the typed label and group) or **Delete selected**.

**Images without objects:** tick **reviewed (no more objects)** to include the image as a distractor. Images with boxes count as reviewed automatically.

The table at the bottom lists every label with its number of boxes and images. Its *warning* column flags labels with only one box (they can't be found again) and labels that look like another label.

In [ ]:
app = LabelingApp(TESTSET_DIR) if TESTSET_DIR.is_dir() else None
app.widget if app else None

## Check the label set

The same overview as at the bottom of the app, as a table you can sort and filter. Run this cell again after a labeling session.

Things to fix before evaluating:
- **`only 1 box`:** an object needs at least two boxes (two photos) to be evaluated. One is used as the query and the others must be found. Add another photo, or accept that it only serves as a distractor.
- **`similar to …`:** probably the same object under two spellings. Relabel one of them in the app.

In [ ]:
if app:
    print(app.annotations.summary())
    stats = app.annotations.label_stats()
    display(stats)
    problems = stats[stats["warning"] != ""]
    print(f"{len(problems)} label(s) with a warning" if len(problems) else "No warnings.")

## Preview the test set as the evaluation will see it

This turns the annotations into test items (one per box, plus the reviewed images without boxes as distractors) and makes the same reproducible split the evaluation uses:
- for every label, one box is the **query**;
- the other boxes of that label, all other boxes and the distractors form the **gallery**.

Labels with a single box can't be split and are left out, with a validator message. Nothing is computed here; this only shows whether the set is ready.

In [ ]:
if app:
    records = app.annotations.to_records()
    print(validate_dataset(parse_dataset(records)).summary())
    if records:
        split = split_dataset(records, seed=0, strict=False)
        print("split:", split.summary())
        for q in split.queries[:10]:
            print(f"  query {q.item_id}  label={q.label}  group={q.group}")